# Exercise 2.3: Schema and Partition Evolution

In this exercise, you'll learn how Apache Iceberg handles table evolution using the NYC Yellow Taxi dataset:
- **Schema Evolution**: Add, remove, and rename columns without rewriting data
- **Field IDs**: Understand how Iceberg uses internal IDs to prevent data resurrection (accidentally restoring deleted column data when a column name is reused)
- **Partition Evolution**: Change partitioning strategies over time

## Learning Objectives
- Add and drop columns using metadata-only operations
- See how field IDs prevent accidentally resurrecting deleted data
- Rename columns without breaking existing data
- Change partition specifications without rewriting data
- Understand the impact of partition evolution on query performance

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("SchemaAndPartitionEvolution") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.evolution")
print("Namespace 'evolution' created!")

## Helper Functions: Iceberg Java API via py4j

Spark SQL does not expose some Iceberg internals like field IDs or partition spec details. We'll need these to inspect Iceberg's internal field IDs. You'll see why they matter in Part 2, where they prevent a subtle bug called data resurrection. We use the py4j bridge to call `Spark3Util.loadIcebergTable`, which returns the Java `Table` object. This gives us direct access to the Iceberg SDK's Java interface, where lower-level table metadata (schemas with field IDs, full partition spec history, column defaults) is available programmatically.

> **Note:** You do not need to understand the implementation of these helpers. They look more complex than typical Python code because they are calling Java APIs from Python via the py4j bridge. Just treat them as utilities. The rest of the notebook will call them by name.

In [ ]:
def load_iceberg_table(table_name):
    """Load an Iceberg Table object via the Spark3Util Java bridge."""
    jvm = spark._jvm
    return jvm.org.apache.iceberg.spark.Spark3Util.loadIcebergTable(
        spark._jsparkSession, table_name
    )

def show_iceberg_schema(table_name, highlight_field_id=None, highlight_label=None):
    """Print the Iceberg schema with field IDs.
    
    Optionally highlight a specific field ID with a label.
    If the highlighted field ID is not present, a note is printed at the end.
    """
    table = load_iceberg_table(table_name)
    schema = table.schema()
    found_highlight = False
    print(f"{'ID':>4}  {'Name':<30} {'Type':<12} {'Optional'}")
    print("-" * 60)
    for col in schema.columns():
        marker = ""
        if highlight_field_id is not None and col.fieldId() == highlight_field_id:
            found_highlight = True
            marker = f"  <-- {highlight_label or 'highlighted'}"
        print(f"{col.fieldId():>4}  {col.name():<30} {str(col.type()):<12} {col.isOptional()}{marker}")
    if highlight_field_id is not None and not found_highlight:
        print(f"\n  * Field ID {highlight_field_id} is absent: {highlight_label or 'not in schema'}")

def show_partition_specs(table_name, highlight_spec_id=None, highlight_label=None):
    """Print all partition specs for an Iceberg table, marking the current one.
    
    Optionally highlight a specific spec ID with an additional label.
    """
    table = load_iceberg_table(table_name)
    specs = table.specs()
    current_id = table.spec().specId()
    print(f"{'Spec ID':<10} {'Fields':<50} {'Status'}")
    print("-" * 70)
    for spec_id in sorted(specs):
        spec = specs[spec_id]
        fields = []
        for f in spec.fields():
            fields.append(f"{f.transform()}({f.name()})")
        field_str = ", ".join(fields) if fields else "(unpartitioned)"
        tags = []
        if spec_id == current_id:
            tags.append("current")
        if highlight_spec_id is not None and spec_id == highlight_spec_id:
            tags.append(highlight_label or "highlighted")
        status = f"  <-- {', '.join(tags)}" if tags else ""
        print(f"{spec_id:<10} {field_str:<50} {status}")

def add_column_with_default(table_name, col_name, iceberg_type, default_value):
    """Add a column with an initial default using the Iceberg Java API.
    
    The Iceberg Spark integration does not yet support initial defaults
    in ALTER TABLE ADD COLUMN, so we call the Java UpdateSchema API
    directly. Requires format-version 3.
    
    The initial default is applied retroactively: existing rows that were
    written before the column existed will return this value instead of NULL.
    """
    jvm = spark._jvm
    Literal = jvm.org.apache.iceberg.expressions.Literal
    table = load_iceberg_table(table_name)
    default_literal = Literal.of(default_value)
    table.updateSchema().addColumn(col_name, iceberg_type, default_literal).commit()
    spark.sql(f"REFRESH TABLE {table_name}")

def show_column_defaults(table_name):
    """Print columns that have initial or write defaults."""
    table = load_iceberg_table(table_name)
    schema = table.schema()
    has_any = False
    for col in schema.columns():
        initial = col.initialDefault()
        write = col.writeDefault()
        if initial is not None or write is not None:
            has_any = True
            print(f"  {col.name()}: initialDefault={initial}, writeDefault={write}")
    if not has_any:
        print("  (no columns have defaults)")

print("Iceberg Java API helpers loaded!")

## Download NYC Taxi Data

We'll use NYC Yellow Taxi trip data for **June through September 2023**. If you've already run earlier exercises, the files may already be in MinIO.

In [ ]:
import boto3
from botocore.client import Config
import urllib.request
import os

s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket = "warehouse"

for month in [6, 7, 8, 9]:
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    key = f"raw/{filename}"
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        print(f"{filename} already in MinIO, skipping download")
    except:
        local_path = f"/tmp/{filename}"
        print(f"Downloading {filename} (~45MB)...")
        urllib.request.urlretrieve(base_url.format(month), local_path)
        s3_client.upload_file(local_path, bucket, key)
        os.remove(local_path)
        print(f"  Uploaded to s3a://{bucket}/{key}")

print("\nAll taxi data ready in MinIO!")

## Part 1: Schema Evolution - Adding Columns

Schema evolution in Iceberg is a metadata-only operation. Adding, dropping, or renaming columns never rewrites data files. Unlike traditional databases where schema changes can require expensive table rewrites (e.g., `ALTER TABLE ADD COLUMN` with a non-null default), Iceberg schema evolution is always instant. The operation updates the schema definition in the metadata file, and query engines apply it at read time.

### Create Table from June Data

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.evolution.nyc_taxi")

spark.sql("""
    CREATE TABLE polaris.evolution.nyc_taxi
    USING iceberg
    TBLPROPERTIES ('format-version' = '3')
    AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

count = spark.sql("SELECT COUNT(*) FROM polaris.evolution.nyc_taxi").collect()[0][0]
print(f"Table created with {count:,} trips (format-version 3)")

In [ ]:
print("Initial schema:")
spark.sql("DESCRIBE polaris.evolution.nyc_taxi").show(truncate=False)

### Add a Column - Instant Operation

In [ ]:
start = time.time()

spark.sql("""
    ALTER TABLE polaris.evolution.nyc_taxi
    ADD COLUMN tip_percentage DOUBLE
""")

elapsed = time.time() - start
print(f"Column added in {elapsed:.3f} seconds (metadata-only operation!)")

In [ ]:
print("Schema after adding tip_percentage:")
spark.sql("DESCRIBE polaris.evolution.nyc_taxi").show(truncate=False)

In [ ]:
print("Old rows have NULL for the new column:")
spark.sql("""
    SELECT VendorID, fare_amount, tip_amount, tip_percentage
    FROM polaris.evolution.nyc_taxi
    LIMIT 5
""").show()

### Insert New Data with the New Column Populated

In [ ]:
spark.sql("""
    INSERT INTO polaris.evolution.nyc_taxi
    SELECT *, ROUND(tip_amount / NULLIF(fare_amount, 0) * 100, 2) as tip_percentage
    FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-07.parquet`
""")

print("July data inserted with tip_percentage populated!")

In [ ]:
print("New rows have tip_percentage, old rows still have NULL:")
spark.sql("""
    SELECT VendorID, fare_amount, tip_amount, tip_percentage
    FROM polaris.evolution.nyc_taxi
    WHERE tip_percentage IS NOT NULL
    LIMIT 5
""").show()

null_count = spark.sql("SELECT COUNT(*) FROM polaris.evolution.nyc_taxi WHERE tip_percentage IS NULL").collect()[0][0]
filled_count = spark.sql("SELECT COUNT(*) FROM polaris.evolution.nyc_taxi WHERE tip_percentage IS NOT NULL").collect()[0][0]
print(f"Rows with NULL tip_percentage (June): {null_count:,}")
print(f"Rows with tip_percentage (July):      {filled_count:,}")

### Add a Column with an Initial Default (Java API)

When you add a column with Spark SQL, existing rows get `NULL` for the new column. Iceberg supports **initial defaults** that apply retroactively to all existing rows, but the Iceberg Spark integration does not yet support this feature. We use the Iceberg Java API directly via the `add_column_with_default` helper.

Initial defaults require **format-version 3**. Iceberg has three format versions: V1 was the original spec, V2 added row-level delete support (the ability to delete or update individual rows within files rather than only appending or dropping entire files, which we'll explore in E3.1), and V3 (the latest) adds features like initial column defaults and default value expressions. Most production deployments currently use V2; V3 is recommended for new tables.

In [ ]:
Types = spark._jvm.org.apache.iceberg.types.Types

add_column_with_default(
    "polaris.evolution.nyc_taxi",
    "surge_multiplier",
    Types.DoubleType.get(),
    spark._jvm.java.lang.Double(1.0)
)

print("Column 'surge_multiplier' added with initial default = 1.0")
print()
print("Existing rows (written before the column existed) return the default:")
spark.sql("""
    SELECT VendorID, fare_amount, surge_multiplier
    FROM polaris.evolution.nyc_taxi
    LIMIT 5
""").show()

In [ ]:
print("Compare with tip_percentage (added via Spark SQL, no default):")
spark.sql("""
    SELECT VendorID, tip_percentage, surge_multiplier
    FROM polaris.evolution.nyc_taxi
    LIMIT 5
""").show()

print("Column defaults stored in metadata:")
show_column_defaults("polaris.evolution.nyc_taxi")

`tip_percentage` shows `NULL` for old rows because it was added without a default. `surge_multiplier` shows `1.0` because the initial default fills in the value at read time without rewriting any data files.

In [ ]:
spark.sql("ALTER TABLE polaris.evolution.nyc_taxi DROP COLUMN surge_multiplier")
print("Dropped surge_multiplier (cleanup for next section)")

### Try It: Evolve the Schema

Add a column of your choice to the `nyc_taxi` table (e.g., `trip_type STRING`, `is_shared_ride BOOLEAN`). Insert a few rows with a value for the new column, then query the table to see how old rows show `NULL` while new rows have the value. When you're done, drop the column to clean up.

If you want a challenge, try adding the column with an initial default using the `add_column_with_default` helper so that old rows get a value too.

In [ ]:
# my_column = "???"
# my_type = "STRING"  # or BOOLEAN, INT, DOUBLE, etc.

# Add the column
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi ADD COLUMN {my_column} {my_type}")
# show_iceberg_schema("polaris.evolution.nyc_taxi")

# Query to see NULL for old rows
# spark.sql(f"SELECT VendorID, fare_amount, {my_column} FROM polaris.evolution.nyc_taxi LIMIT 5").show()

# Clean up when done
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi DROP COLUMN {my_column}")

## Part 2: Field IDs and Data Resurrection Prevention

Iceberg tracks columns by internal field IDs, not by name. This prevents a subtle bug: if you drop a column and later re-add one with the same name, old data from the dropped column does not come back.

### View Field IDs

In [ ]:
print("Schema with Iceberg field IDs:")
show_iceberg_schema(
    "polaris.evolution.nyc_taxi",
    highlight_field_id=7,
    highlight_label="we will drop this column next"
)

### Drop a Column

In [ ]:
spark.sql("""
    ALTER TABLE polaris.evolution.nyc_taxi
    DROP COLUMN store_and_fwd_flag
""")

print("store_and_fwd_flag column dropped!")

In [ ]:
print("Schema after dropping store_and_fwd_flag (note: field ID 7 is now retired):")
show_iceberg_schema(
    "polaris.evolution.nyc_taxi",
    highlight_field_id=7,
    highlight_label="retired (was store_and_fwd_flag), IDs skip from 6 to 8"
)

### Re-add Column with the Same Name

This is where field IDs shine. The old data will not be resurrected.

In [ ]:
spark.sql("""
    ALTER TABLE polaris.evolution.nyc_taxi
    ADD COLUMN store_and_fwd_flag STRING
""")

print("store_and_fwd_flag column re-added!")

In [ ]:
print("Field IDs after re-adding store_and_fwd_flag:")
show_iceberg_schema(
    "polaris.evolution.nyc_taxi",
    highlight_field_id=22,
    highlight_label="new ID for re-added column (not 7!)"
)

print("\nOld store_and_fwd_flag data is NOT resurrected:")
spark.sql("""
    SELECT VendorID, fare_amount, store_and_fwd_flag
    FROM polaris.evolution.nyc_taxi
    LIMIT 5
""").show()

Notice that the re-added `store_and_fwd_flag` column got a **new** field ID (not 7). Old data files still have field 7, but nothing in the current schema maps to it, so the old data is **not** resurrected.

**What happened?**
- The original `store_and_fwd_flag` was field ID **7**
- When we dropped it, field ID 7 was retired (you can see the gap in the ID sequence)
- The new `store_and_fwd_flag` received a fresh field ID (the next available number)
- Old data files still contain the original column under field ID 7, but the current schema maps the name to the new ID
- Result: old data is invisible, preventing accidental data resurrection

### Try It: Test Data Resurrection Prevention

Drop a column (e.g., `extra`), note its field ID, then re-add it with the same name. Use `show_iceberg_schema` to confirm it received a new field ID. Query the table. Does the old data come back?

In [ ]:
# my_column = "extra"  # pick any column

# Check the field ID before dropping
# show_iceberg_schema("polaris.evolution.nyc_taxi")

# Drop and re-add
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi DROP COLUMN {my_column}")
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi ADD COLUMN {my_column} DOUBLE")

# Check the new field ID. It should be different!
# show_iceberg_schema("polaris.evolution.nyc_taxi")

# Query: old values should be NULL (not resurrected)
# spark.sql(f"SELECT VendorID, {my_column} FROM polaris.evolution.nyc_taxi LIMIT 5").show()

# Clean up: drop again so the notebook continues cleanly
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi DROP COLUMN {my_column}")

## Part 3: Renaming Columns

In [ ]:
spark.sql("""
    ALTER TABLE polaris.evolution.nyc_taxi
    RENAME COLUMN fare_amount TO base_fare
""")

print("Column renamed from 'fare_amount' to 'base_fare'")

In [ ]:
print("Schema after rename (field ID unchanged, only the name mapping changed):")
show_iceberg_schema(
    "polaris.evolution.nyc_taxi",
    highlight_field_id=11,
    highlight_label="was 'fare_amount', same field ID"
)

In [ ]:
print("Query using new name:")
spark.sql("""
    SELECT VendorID, base_fare, tip_amount
    FROM polaris.evolution.nyc_taxi
    WHERE base_fare > 100
    LIMIT 10
""").show()

The field ID did not change, only the name mapping. All existing data is still accessible through the new name.

### Try It: Rename and Verify

Rename another column (e.g., `tip_amount` to `gratuity`). Use `show_iceberg_schema` to confirm the field ID is unchanged. Query the table using the new name, then rename it back.

In [ ]:
# old_name = "tip_amount"
# new_name = "gratuity"

# Rename it
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi RENAME COLUMN {old_name} TO {new_name}")
# show_iceberg_schema("polaris.evolution.nyc_taxi")

# Query with the new name
# spark.sql(f"SELECT VendorID, base_fare, {new_name} FROM polaris.evolution.nyc_taxi LIMIT 5").show()

# Rename back to keep the notebook consistent
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi RENAME COLUMN {new_name} TO {old_name}")

## Part 4: Partition Evolution

Iceberg lets you change the partition scheme of a table without rewriting existing data. In traditional table formats, changing a table's partitioning requires rewriting all existing data. Iceberg tracks which partition spec each file was written under, so old files keep their original layout and new files use the new spec. Query engines reconcile the different specs automatically at read time.

### Create an Unpartitioned Table

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.evolution.taxi_evolving")

spark.sql("""
    CREATE TABLE polaris.evolution.taxi_evolving
    USING iceberg
    AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

count = spark.sql("SELECT COUNT(*) FROM polaris.evolution.taxi_evolving").collect()[0][0]
print(f"Unpartitioned table created with {count:,} trips")

In [ ]:
print("Files (unpartitioned):")
spark.sql("""
    SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
           spec_id, record_count
    FROM polaris.evolution.taxi_evolving.files
""").show(truncate=False)

print()
show_partition_specs(
    "polaris.evolution.taxi_evolving",
    highlight_spec_id=0,
    highlight_label="initial (unpartitioned)"
)

### Add Day Partitioning

In [ ]:
spark.sql("""
    ALTER TABLE polaris.evolution.taxi_evolving
    ADD PARTITION FIELD days(tpep_pickup_datetime)
""")

print("Day partitioning added! New writes will be partitioned by day.")

### Insert New Data - Now Partitioned

In [ ]:
spark.sql("""
    INSERT INTO polaris.evolution.taxi_evolving
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-07.parquet`
""")

print("July data inserted (will be day-partitioned)!")

In [ ]:
print("Files after adding day partitioning (mix of spec 0 and spec 1):")
spark.sql("""
    SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
           spec_id, partition, record_count
    FROM polaris.evolution.taxi_evolving.files
    ORDER BY spec_id, partition
""").show(40, truncate=False)

print()
show_partition_specs(
    "polaris.evolution.taxi_evolving",
    highlight_spec_id=1,
    highlight_label="just added: day(pickup)"
)

Old files remain under spec 0 (unpartitioned). New files are written under spec 1 (day-partitioned). The specs table shows the full evolution history.

**Reading the `partition` column:** The partition value shown in the `files` metadata table is a unified tuple that includes a slot for every partition field that has ever existed on the table. The tuple grows each time a new partition field is added, and fields that don't apply to a given file's spec appear as `NULL`.

Let's walk through this step by step:

1. **After spec 1 is added** (day partitioning), the tuple has **one slot**: `{day}`.
   - Spec 0 files (unpartitioned): `{NULL}` because the day field doesn't apply
   - Spec 1 files (day-partitioned): `{2023-07-01}` where the day value is filled in

2. **After spec 2 is added** (month partitioning), a **second slot** appears: `{day, month}`.
   - Spec 0 files: `{NULL, NULL}` because neither field applies
   - Spec 1 files: `{2023-07-01, NULL}` with day filled, month N/A
   - Spec 2 files: `{NULL, 643}` with day N/A, month filled (643 = August 2023, as months since epoch)

Iceberg uses this unified structure so that query engines can compare partition values across files from different specs using a single consistent format.

### Change to Month Partitioning

In [ ]:
spark.sql("""
    ALTER TABLE polaris.evolution.taxi_evolving
    DROP PARTITION FIELD days(tpep_pickup_datetime)
""")

spark.sql("""
    ALTER TABLE polaris.evolution.taxi_evolving
    ADD PARTITION FIELD months(tpep_pickup_datetime)
""")

print("Partition changed from days to months!")

In [ ]:
spark.sql("""
    INSERT INTO polaris.evolution.taxi_evolving
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-08.parquet`
""")

print("August data inserted (will be month-partitioned)!")

In [ ]:
print("Files with three partition specs (5 per spec):")
spark.sql("""
    SELECT filename, spec_id, partition, record_count FROM (
        SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
               spec_id, partition, record_count,
               ROW_NUMBER() OVER (PARTITION BY spec_id ORDER BY partition) as rn
        FROM polaris.evolution.taxi_evolving.files
    ) WHERE rn <= 5
    ORDER BY spec_id, partition
""").show(truncate=False)

print()
show_partition_specs(
    "polaris.evolution.taxi_evolving",
    highlight_spec_id=2,
    highlight_label="just added: month(pickup)"
)

### Revert to Day Partitioning (Spec Reuse)

What happens if we switch back to day partitioning? Iceberg does **not** create a new spec. It recognizes that spec 1 already represents `days(tpep_pickup_datetime)` and reuses it.

In [ ]:
spark.sql("""
    ALTER TABLE polaris.evolution.taxi_evolving
    DROP PARTITION FIELD months(tpep_pickup_datetime)
""")

spark.sql("""
    ALTER TABLE polaris.evolution.taxi_evolving
    ADD PARTITION FIELD days(tpep_pickup_datetime)
""")

print("Switched back to day partitioning!")

In [ ]:
spark.sql("""
    INSERT INTO polaris.evolution.taxi_evolving
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-09.parquet`
""")

print("September data inserted (day-partitioned again)!")

In [ ]:
print("Files after reverting to day partitioning (5 per spec):")
spark.sql("""
    SELECT filename, spec_id, partition, record_count FROM (
        SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
               spec_id, partition, record_count,
               ROW_NUMBER() OVER (PARTITION BY spec_id ORDER BY partition) as rn
        FROM polaris.evolution.taxi_evolving.files
    ) WHERE rn <= 5
    ORDER BY spec_id, partition
""").show(truncate=False)

print()
show_partition_specs(
    "polaris.evolution.taxi_evolving",
    highlight_spec_id=1,
    highlight_label="reused (no new spec created!)"
)

Notice there are still only **three** spec IDs (0, 1, 2). The new day-partitioned files land under **spec 1**, the same spec that was used earlier. Iceberg tracks partition specs by their definition, so reverting to a previous scheme reuses the existing spec rather than creating a redundant new one.

### Try It: Try a Different Partition Transform

Iceberg supports several partition transforms: `years`, `months`, `days`, `hours`, `bucket(N, col)`, and `truncate(N, col)`. Here's what the less obvious ones do:
- **`bucket(N, col)`** hashes column values into N fixed groups, similar to hash partitioning in traditional databases. Useful for high-cardinality columns where temporal transforms don't apply.
- **`truncate(N, col)`** groups values by their first N characters (for strings) or by rounding to the nearest N (for numbers).

Try adding a different partition to `taxi_evolving`, for example, `hours(tpep_pickup_datetime)` or `bucket(16, PULocationID)`. Insert some data, then inspect the files and partition specs to see the result. Use `show_partition_specs` to track how many specs the table has accumulated. You can also verify your partition change by inspecting the `files` metadata table. If the new files have the expected partition values, the transform is working.

In [ ]:
# my_transform = "hours(tpep_pickup_datetime)"  # or bucket(16, PULocationID), etc.

# First, drop the current partition field
# spark.sql("ALTER TABLE polaris.evolution.taxi_evolving DROP PARTITION FIELD days(tpep_pickup_datetime)")

# Add your partition transform
# spark.sql(f"ALTER TABLE polaris.evolution.taxi_evolving ADD PARTITION FIELD {my_transform}")
# show_partition_specs("polaris.evolution.taxi_evolving")

# Insert some data and inspect the files
# spark.sql("""
#     INSERT INTO polaris.evolution.taxi_evolving
#     SELECT * FROM polaris.evolution.taxi_evolving LIMIT 1000
# """)
# spark.sql("""
#     SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
#            spec_id, partition, record_count
#     FROM polaris.evolution.taxi_evolving.files
#     ORDER BY spec_id DESC
#     LIMIT 10
# """).show(truncate=False)

## Cleanup

In [ ]:
# Optional: Drop tables to start fresh
# spark.sql("DROP TABLE IF EXISTS polaris.evolution.nyc_taxi")
# spark.sql("DROP TABLE IF EXISTS polaris.evolution.taxi_evolving")
# print("Tables dropped!")